In [ ]:
#!/usr/bin/env python3
"""
Extract ELECT and VDW interaction-energy components from NAMD log files.

Calculation:
    ΔE = E_complex - E_receptor - E_ligand

Example:
    python extract_interaction_energy.py \
        --base-dir /path/to/mmgbsa_control_12_23 \
        --systems mmgbsa606 mmgbsa687 mmgbsa688 mmgbsa688_2 mmgbsa721 \
        --replicas 1 2 3 \
        --output all_systems_interaction_energy.npz
"""

import argparse
from pathlib import Path

import numpy as np


ENERGY_TERMS = ("ELECT", "VDW")


def parse_arguments():
    """讀取命令列參數。"""

    parser = argparse.ArgumentParser(
        description="Extract ELECT and VDW interaction energies from NAMD logs."
    )

    parser.add_argument(
        "--base-dir",
        required=True,
        type=Path,
        help="Directory containing system_replica folders.",
    )

    parser.add_argument(
        "--systems",
        required=True,
        nargs="+",
        help="System names without the _repX suffix.",
    )

    parser.add_argument(
        "--replicas",
        nargs="+",
        type=int,
        default=[1, 2, 3],
        help="Replica numbers. Default: 1 2 3.",
    )

    parser.add_argument(
        "--output",
        required=True,
        type=Path,
        help="Output NPZ file.",
    )

    return parser.parse_args()


def find_log_file(directory, keywords):
    """
    根據關鍵字尋找 NAMD log。

    如果有多個符合檔案，會顯示警告並使用排序後的第一個。
    """

    log_files = sorted(directory.glob("*.log"))

    matched_files = [
        path
        for path in log_files
        if any(keyword.lower() in path.name.lower() for keyword in keywords)
    ]

    if not matched_files:
        return None

    if len(matched_files) > 1:
        print(
            f"  [警告] {directory.name} 找到多個符合檔案："
            f"{', '.join(path.name for path in matched_files)}"
        )
        print(f"  使用：{matched_files[0].name}")

    return matched_files[0]


def extract_energy_from_log(file_path, target_terms=ENERGY_TERMS):
    """從 NAMD log 的 ENERGY 欄位提取指定能量項。"""

    energy_data = {
        term: []
        for term in target_terms
    }

    column_indices = {}

    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
            errors="replace",
        ) as handle:

            for line in handle:
                fields = line.split()

                if not fields:
                    continue

                if fields[0] == "ETITLE:":
                    column_indices = {
                        term: fields.index(term)
                        for term in target_terms
                        if term in fields
                    }

                elif fields[0] == "ENERGY:" and column_indices:
                    for term in target_terms:
                        if term not in column_indices:
                            continue

                        index = column_indices[term]

                        if index >= len(fields):
                            continue

                        try:
                            energy_data[term].append(
                                float(fields[index])
                            )
                        except ValueError:
                            continue

    except OSError as error:
        print(f"  [錯誤] 無法讀取 {file_path}: {error}")
        return None

    missing_terms = [
        term
        for term in target_terms
        if not energy_data[term]
    ]

    if missing_terms:
        print(
            f"  [錯誤] {file_path.name} 缺少有效欄位："
            f"{', '.join(missing_terms)}"
        )
        return None

    return {
        term: np.asarray(values, dtype=float)
        for term, values in energy_data.items()
    }


def find_system_directories(base_dir, systems, replicas):
    """尋找實際存在的 system_replica 資料夾。"""

    directories = []

    print("正在掃描系統資料夾……")

    for system in systems:
        for replica in replicas:
            directory = base_dir / f"{system}_rep{replica}"

            if directory.is_dir():
                directories.append(directory)
                print(f"  [OK] {directory.name}")
            else:
                print(f"  [缺少] {directory.name}")

    print(f"共找到 {len(directories)} 個資料夾。")

    return directories


def calculate_interaction_energy(
    complex_data,
    receptor_data,
    ligand_data,
):
    """
    計算 ΔE = E_complex - E_receptor - E_ligand。

    所有能量項會裁切至共同的最短 frame 數。
    """

    all_lengths = []

    for dataset in (
        complex_data,
        receptor_data,
        ligand_data,
    ):
        for term in ENERGY_TERMS:
            all_lengths.append(len(dataset[term]))

    frame_count = min(all_lengths)

    if frame_count == 0:
        return None, 0

    delta_energy = {}

    for term in ENERGY_TERMS:
        delta_energy[term] = (
            complex_data[term][:frame_count]
            - receptor_data[term][:frame_count]
            - ligand_data[term][:frame_count]
        )

    return delta_energy, frame_count


def process_system_directory(directory):
    """處理單一 system_replica 資料夾。"""

    print(f"\n正在處理：{directory.name}")

    log_paths = {
        "complex": find_log_file(
            directory,
            ["complex"],
        ),
        "receptor": find_log_file(
            directory,
            ["target", "receptor"],
        ),
        "ligand": find_log_file(
            directory,
            ["ligand"],
        ),
    }

    missing_logs = [
        component
        for component, path in log_paths.items()
        if path is None
    ]

    if missing_logs:
        print(
            f"  [跳過] 缺少 log："
            f"{', '.join(missing_logs)}"
        )
        return None

    for component, path in log_paths.items():
        print(f"  {component}: {path.name}")

    energy_data = {
        component: extract_energy_from_log(path)
        for component, path in log_paths.items()
    }

    if any(value is None for value in energy_data.values()):
        print("  [跳過] 能量資料提取失敗。")
        return None

    delta_energy, frame_count = calculate_interaction_energy(
        complex_data=energy_data["complex"],
        receptor_data=energy_data["receptor"],
        ligand_data=energy_data["ligand"],
    )

    if delta_energy is None:
        print("  [跳過] 沒有共同的有效 frame。")
        return None

    print(f"  Frames: {frame_count}")

    for term in ENERGY_TERMS:
        print(
            f"  Δ{term}: "
            f"{np.mean(delta_energy[term]):.3f} kcal/mol"
        )

    return delta_energy


def main():
    """主程式。"""

    args = parse_arguments()

    if not args.base_dir.is_dir():
        raise NotADirectoryError(
            f"找不到基礎資料夾：{args.base_dir}"
        )

    system_directories = find_system_directories(
        base_dir=args.base_dir,
        systems=args.systems,
        replicas=args.replicas,
    )

    if not system_directories:
        raise RuntimeError("沒有找到任何可處理的系統資料夾。")

    output_data = {}
    processed_count = 0

    for directory in system_directories:
        delta_energy = process_system_directory(directory)

        if delta_energy is None:
            continue

        for term in ENERGY_TERMS:
            output_data[
                f"{directory.name}_Delta_{term}"
            ] = delta_energy[term]

        processed_count += 1

    if not output_data:
        raise RuntimeError("沒有任何有效資料可以儲存。")

    args.output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        args.output,
        **output_data,
    )

    print("\n" + "=" * 60)
    print(f"處理完成：{processed_count} 個 system-replica")
    print(f"輸出檔案：{args.output}")


if __name__ == "__main__":
    main()